# Moving from scArches

This tutorial is for users with scArches scPoli code or trained scArches models:

1. how scArches calls translate to scpolivi;
2. loading a model trained and saved with scArches, **without installing scArches**;
3. continuing with it in scpolivi: projection, query mapping, saving in scvi-tools format.

The checkpoint used here was produced by the scArches surgery tutorial
(`scripts/make_golden_pancreas.py`, run in an environment with scArches).

## Translating code

The data arguments move from the constructor to `setup_anndata`, as in every scvi-tools
model. Everything else keeps its scArches name.

```python
# scArches
from scarches.models.scpoli import scPoli
model = scPoli(adata=ref, condition_keys="study", cell_type_keys="cell_type",
               embedding_dims=5, recon_loss="nb")
model.train(n_epochs=50, pretraining_epochs=40, eta=5)
query_model = scPoli.load_query_data(adata=query, reference_model=model, labeled_indices=[])

# scpolivi
from scpolivi import SCPOLI
SCPOLI.setup_anndata(ref, condition_keys="study", cell_type_keys="cell_type")
model = SCPOLI(ref, embedding_dims=5, recon_loss="nb")
model.train(n_epochs=50, pretraining_epochs=40, eta=5)
query_model = SCPOLI.load_query_data(adata=query, reference_model=model, labeled_indices=[])
```

| scArches | scpolivi |
|---|---|
| `model.trainer.logs["val_loss"]` | `model.history["validation_loss"]` |
| `model.model` | `model.module` |
| `scPoli.load(dir, adata)` for scArches files | `SCPOLI.from_scarches(dir, adata)` |

In [1]:
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import scvi

from scpolivi import SCPOLI

warnings.filterwarnings("ignore")
scvi.settings.seed = 0
scvi.settings.verbosity = logging.WARNING
sc.set_figure_params(figsize=(4.5, 4.5), frameon=False, dpi=80)

## Loading a scArches checkpoint

A scArches model directory contains `attr.pkl`, `model_params.pt` and `var_names.csv`.
`from_scarches` reads them with plain pickle/torch and converts the weights and prototypes.
The AnnData must have the model's genes in the same order, and its condition columns.

In [2]:
SCARCHES_MODEL = Path("data/golden/pancreas_seed0/ref")   # a scArches scPoli.save() directory
sorted(p.name for p in SCARCHES_MODEL.iterdir())

['attr.pkl', 'model_params.pt', 'var_names.csv']

In [3]:
adata = sc.read("data/pancreas_sparse.h5ad")
reference = adata[~adata.obs.study.isin(["celseq", "celseq2"])].copy()
reference = reference[reference.obs.cell_type != "alpha"].copy()
query = adata[adata.obs.study.isin(["celseq", "celseq2"])].copy()

model = SCPOLI.from_scarches(SCARCHES_MODEL, reference)
model

SCPOLI model: conditions ['study'] [7], cell types ['cell_type'] (13 prototypes), hidden [64], latent 10, 
recon_loss nb
Training status: Trained
Model's adata is minified?: False

In [4]:
print("conditions:", model.conditions_)
print("cell types:", list(model.cell_types_))

conditions: {'study': ['fluidigmc1', 'inDrop1', 'inDrop2', 'inDrop3', 'inDrop4', 'smarter', 'smartseq2']}
cell types: ['gamma', 'acinar', 'delta', 'beta', 'ductal', 'endothelial', 'activated_stellate', 'schwann', 'mast', 'macrophage', 'epsilon', 'quiescent_stellate', 't_cell']


Check that the conversion is exact. The scArches run also saved the latent means of the
reference cells, as computed by scArches (`outputs.npz`). The imported model should
reproduce them:

In [5]:
scarches_latent = np.load(SCARCHES_MODEL.parent / "outputs.npz")["latent_source"]
latent = model.get_latent(reference, mean=True)
print("max |difference| to scArches latent:", float(np.abs(latent - scarches_latent).max()))

res = model.classify(reference, use_mean=True)["cell_type"]
print("reference cells classified as their annotated type:",
      round(float((res["preds"] == reference.obs.cell_type.values).mean()), 3))

max |difference| to scArches latent: 4.76837158203125e-07
reference cells classified as their annotated type: 0.978


## Continuing in scpolivi

The imported model is an ordinary scpolivi model. It can map a query, be saved in scvi-tools
format, and use the scvi-tools tools.

In [6]:
query_model = SCPOLI.load_query_data(query, reference_model=model, labeled_indices=[])
query_model.train(n_epochs=50, pretraining_epochs=40, eta=10)
pred = query_model.classify(query, use_mean=True)["cell_type"]["preds"]
known = (query.obs.cell_type != "alpha").to_numpy()
print("query label transfer:", round(float((pred[known] == query.obs.cell_type.values[known]).mean()), 3))

Epoch 50/50: 100%|██████████| 50/50 [00:26<00:00,  1.86it/s, v_num=1, train_loss=1.49e+3]


query label transfer: 0.976


In [7]:
model.save("results/tutorials/from_scarches_reference", overwrite=True, save_anndata=True)
SCPOLI.load("results/tutorials/from_scarches_reference")

SCPOLI model: conditions ['study'] [7], cell types ['cell_type'] (13 prototypes), hidden [64], latent 10, 
recon_loss nb
Training status: Trained
Model's adata is minified?: False

## Behaviour differences to keep in mind

scpolivi keeps scArches' behaviour except where scArches is broken. The most relevant
differences:

- `classify(use_mean=True)` is recommended. The default `False` classifies a random latent
  sample, as scArches does.
- `seed` defaults to `None` (`scvi.settings.seed` is used). scArches reseeds with 2020.
- `reload_best=True` restores weights and prototypes. scArches only reverts the prototypes.
- The `bernoulli` and `nb_dist` losses work. They crash in scArches.
- Projection checks gene order, and cell-type columns are optional.

The full list is on the *Coming from scArches* page.